# Datamine RECMODEL Process Example

This notebook demonstrates how to configure and run the **`recmodel`** process wrapper in `dmstudio`.

## Process Description

## RECMODEL

RECMODEL is used to compare the tonnes and grades of pairs of groups of cells in a block model with the same value of two ID fields. An example of how it might be used is to compare the optimised pushbacks defined in a block model against designed pushbacks defined by another set of field values.

### Input Files:

* **model** (Block model):
Input model file for reconciliation. It must contain a field identifying the planned
cells to be reconciled with the wireframe volumes. If the model has been created by
Studio NPVS or Studio NPVS+ this field may be the optimally planned pushback identifier

## **PSB_PIT**.

Required=Yes

### Output Files:

* **results** (Results):
Output results file containing the reserve comparisons. This contains up to 5 records
for every separate reconciled volume: Total Planned, Total Mined, Planned and Mined,
Planned Only and Mined Only. Volumes are defined by the **PLANNED** and **MINED** fields
and can be further broken down by the **KEY1** field and **BENCH** parameter.
Required=Yes

### Fields:

* **planned** (Any : MODEL):
Field in **MODEL** file used to group the planned blocks. If comparing wireframe
designs with pushback reserves in a Studio NPVS or Studio NPVS+ model this field may be

## **PSB_PIT**.

Default=Undefined
Required=Yes

* **mined** (Any : WIRETR, MODEL):
Field in the **WIRETR** file defining the volumes to be compared to the corresponding
**PLANNED** block model cells.
Default=Undefined
Required=Yes

* **key1** (Any : MODEL):
Optional Key field in the **MODEL** file used to categorize results (for example, a
rock type field).
Default=Undefined
Required=No

* **density** (Any : MODEL):
Density field in the **MODEL** file used to calculate tonnages.
Default=DENSITY
Required=No

* **grades** (Undefined : Undefined):
Grade field in the model file
Default=Undefined
Required=No

### Parameters:

* **value**:
Value of **PLANNED** and **MINED** fields to compare. If undefined or zero then all
values of **MINED** field will be compared.
Range=Undefined
Values=Undefined
Default=Undefined
Required=No

* **modltype**:
Type of wireframe model to be filled; one of the following options, with default of (1)
:-
Range=
Values=
Default=Solid 3d, interior to be filled with cells
Required=No

* **factor**:
Scaling factor to adjust the units of the Volume and Tonnage in the output files.
Volume and Tonnage are divided by this factor.
Range=Undefined
Values=Undefined
Default=1
Required=No

* **setabsnt**:
Set to 1 to allow **[TONGRAD](<tongrad.md>)** to internally reset absent grade and
density values. If this is used, absent grade values are set to their default values. If
the default value is absent grade values are set to zero. If density values are absent
the default **DENSITY** parameter value is used."
Range=0, 1
Values=0, 1
Default=0
Required=No

* **bench**:
Set to 1 to categorize the reserve comparisons by benches.
Range=
Values=
Default=Do not categorize by benches
Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('recmodel')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute recmodel
print("Running recmodel...")
dm_cmd.recmodel(
    model_i='t_mod1',  # required
    results_o=['t_recmodel_out'],  # required
    planned_f='optional',  # required
    mined_f='optional',  # required
    # key1_f='optional',  # optional
    # density_f='optional',  # optional
    # grades_f=['optional'],  # optional
    # value_p='optional',  # optional
    # modltype_p='Solid 3d, interior to be filled with cells',  # optional
    # factor_p=1,  # optional
    # setabsnt_p=0,  # optional
    # bench_p='Do not categorize by benches',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("recmodel execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_recmodel_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")